# BioEmu Conformational Ensemble Sampling

This notebook demonstrates conformational ensemble sampling using BioEmu. We generate a set of backbone conformations for a short protein sequence, inspect the resulting ensemble, and demonstrate batch sampling across multiple complexes in a single call.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("bioemu")
display_overview("bioemu")
display_docs_section("bioemu", "Background")

# BioEmu

> BioEmu generates protein conformational ensembles using a diffusion generative model trained on [molecular dynamics](https://en.wikipedia.org/wiki/Molecular_dynamics) (MD) simulation data. Given a protein sequence, it produces an ensemble of 3D backbone structures representing the [equilibrium distribution](https://en.wikipedia.org/wiki/Boltzmann_distribution) of conformations the protein adopts -- capturing folded states, alternative conformations, and conformational heterogeneity without running explicit MD simulations.

## Background

Proteins are not static objects. In solution, a protein constantly fluctuates between conformational states, and this dynamics is essential for function: enzyme catalysis, [allosteric regulation](https://en.wikipedia.org/wiki/Allosteric_regulation), molecular recognition, and signal transduction all depend on conformational changes.

Traditional approaches to studying protein dynamics include:

* **Molecular dynamics (MD)**: Physically rigorous but computationally expensive (microseconds of simulation can take weeks of GPU time)
* **[NMR spectroscopy](https://en.wikipedia.org/wiki/Nuclear_magnetic_resonance_spectroscopy_of_proteins)**: Experimental ensemble methods, but limited to small proteins
* **[Normal mode analysis](https://en.wikipedia.org/wiki/Normal_mode)**: Fast but limited to harmonic fluctuations around a single structure

BioEmu takes a fundamentally different approach: it learns the equilibrium conformational distribution directly from MD training data using a score-based diffusion model. Given only a protein sequence, it generates an ensemble of backbone conformations that approximate the [Boltzmann distribution](https://en.wikipedia.org/wiki/Boltzmann_distribution) -- the thermodynamically correct distribution of states the protein visits at equilibrium.

This makes BioEmu orders of magnitude faster than MD while capturing the same large-scale conformational heterogeneity (though it does not model explicit time-dependent dynamics or rare events).

## Available tools

In [2]:
display_available_tools("bioemu")

- **`run_bioemu()`** — Protein conformational ensemble sampling using BioEmu

### `run_bioemu`

BioEmu generates protein conformational ensembles by sampling from a learned distribution over backbone conformations. It accepts one or more single-chain protein complexes and returns an ensemble of backbone structures for each, representing the thermodynamic diversity of conformations that the protein is likely to adopt. BioEmu always runs MSA search via ColabFold before inference, unless pre-computed MSAs are already attached to the input. Multiple complexes can be submitted in a single call; the model processes them together and returns one ensemble per input.

In [3]:
from proto_tools import (
    BioEmuConfig,
    BioEmuInput,
    StructurePredictionComplex,
    run_bioemu,
)

In [4]:
# Display input docs
display_api_reference("bioemu", "input", "run_bioemu")

# Single-chain protein complex for ensemble sampling
complex_ = StructurePredictionComplex(
    chains=[{"sequence": "MVLSPADKTNVKAAW", "entity_type": "protein"}]
)

inputs = BioEmuInput(complexes=[complex_])

**Input** — `BioEmuInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `complexes` | `List[StructurePredictionComplex]` | required | Protein complexes to sample. BioEmu supports monomer-only inputs, so each complex must contain one protein chain. |
| `chains` | `List[Chain]` | required | Chains in the complex. Each chain can be: |
| `msas` | `Dict[string, MSA]` |  | Pre-computed MSAs keyed by protein sequence. Populated by preprocess() or supplied directly. Default: None. |

In [5]:
# Display config docs
display_api_reference("bioemu", "config", "run_bioemu")

# Sample 50 conformations using the v1.1 model with quality filtering enabled
config = BioEmuConfig(
    num_samples=50,
    model_name="bioemu-v1.1",
    filter_samples=True,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `BioEmuConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `num_samples` | `integer` | `500` | Number of conformations to sample per input sequence. |
| `model_name` | `enum` | `bioemu-v1.1` | BioEmu model variant. Available options: `bioemu-v1.0`, `bioemu-v1.1` |
| `filter_samples` | `boolean` | `True` | Whether to filter lower-quality generated samples. |
| `batch_size` | `integer` | `10` | Batch size control for BioEmu internal sampling. |
| `output_dir` | `string` |  | Optional directory for raw BioEmu outputs. |
| `colabfold_search_config` | `ColabfoldSearchConfig` |  | Configuration for ColabFold MSA search. Default: Uses ColabfoldSearchConfig defaults. |
| `verbose` | `boolean` | `False` | Verbose logging toggle (inherited). |
| `device` | `string` | `cuda` | Inference device (inherited). |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [6]:
# Run the sampling
single_result = run_bioemu(inputs, config)

Running run_colabfold_search [00:00]

Running run_bioemu [00:00]

In [7]:
# Display output docs and inspect results
display_api_reference("bioemu", "output", "run_bioemu")

print(f"Ensembles: {len(single_result.ensembles)}")
print(f"Conformations: {len(single_result.ensembles[0].structures)}")

**Output** — `BioEmuOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `ensembles` | `List[StructureEnsemble]` | required | Generated ensembles, one per input complex. |
| `structures` | `List[object]` | required | List of sampled conformational structures. Each Structure represents a single backbone conformation from the ensemble. |
| `sequence` | `string` | required | The input protein sequence. |

Ensembles: 1
Conformations: 48


### Batch sampling across multiple complexes

Multiple complexes can be submitted in a single call, which is more efficient than making separate calls because MSA search and GPU allocation happen only once. Each complex produces its own ensemble in the output.

In [8]:
complex_a = StructurePredictionComplex(
    chains=[{"sequence": "MVLSPADKTNVKAAW", "entity_type": "protein"}]
)
complex_b = StructurePredictionComplex(
    chains=[{"sequence": "GSGSGSGSGSGS", "entity_type": "protein"}]
)

batch_inputs = BioEmuInput(complexes=[complex_a, complex_b])
batch_config = BioEmuConfig(num_samples=20, verbose=False)
batch_result = run_bioemu(batch_inputs, batch_config)

for idx, ensemble in enumerate(batch_result.ensembles):
    print(f"Complex {idx}: {len(ensemble.structures)} conformations")

Running run_colabfold_search [00:00]

Running run_bioemu [00:00]

Complex 0: 20 conformations
Complex 1: 12 conformations


## Export Results

Ensemble outputs can be saved in two formats. PDB format produces a directory tree with one PDB file per conformation, organized by ensemble, which is useful for visualization or as input to downstream tools. JSON format produces a single file containing all conformations, which is convenient for programmatic analysis.

In [9]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

batch_result.export(name="bioemu_batch", export_path=output_dir, file_format="json")
print("Batch ensembles exported to example_output/bioemu_batch.json")

Batch ensembles exported to example_output/bioemu_batch.json
